In [52]:
import os
import copy
import random
import io

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torchvision.utils as vutils

from tqdm.auto import tqdm
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    balanced_accuracy_score,
)
from sklearn.utils.class_weight import compute_class_weight


In [53]:
import mlflow
import mlflow.pytorch

In [54]:
mlflow.set_experiment("MLP_Clasificador_Imagenes")

<Experiment: artifact_location='file:///media/rami/Kingstone/ITBA%20Cursada/RedesNeuro/skin-dataset-classification/mlruns/252422024077438682', creation_time=1782010930521, experiment_id='252422024077438682', last_update_time=1782010930521, lifecycle_stage='active', name='MLP_Clasificador_Imagenes', tags={}>

In [55]:
#from torch.utils.tensorboard import SummaryWriter

# Mejores HP encontrados en la búsqueda optimizada, trial 53.
#BEST_HPARAMS = {
#    "input_size": 64,
#    "batch_size": 64,
#    "lr": 1e-4,
#    "weight_decay": 1e-3,
#    "hidden_sizes": (1024, 512),
#    "dropout": 0.35,
#    "HFlip": 0.5,
#    "VFlip": 0.5,
#    "RBContrast": 0.3,
#    "epochs": 200,
#    "es_patience": 12,
#    "label_smoothing": 0.05,
#    "grad_clip_norm": 1.0,
#    "scheduler_factor": 0.5,
#    "scheduler_patience": 4,
#    "seed": 42,
#    "num_workers": 0,
#}

# Seed fijo: hace la corrida más reproducible.
#def set_seed(seed=42):
#    random.seed(seed)
#    np.random.seed(seed)
#    torch.manual_seed(seed)
#    torch.cuda.manual_seed_all(seed)
#    torch.backends.cudnn.benchmark = False
#    torch.backends.cudnn.deterministic = True

#set_seed(BEST_HPARAMS["seed"])

from torch.utils.tensorboard import SummaryWriter

# Mejores HP encontrados en la búsqueda local actualizada.
# Run: local_037_seed_2063
BEST_HPARAMS = {
    "input_size": 96,
    "batch_size": 64,
    "lr": 5e-5,
    "weight_decay": 1e-3,
    "hidden_sizes": (1536, 768),
    "dropout": 0.35,
    "HFlip": 0.5,
    "VFlip": 0.5,
    "RBContrast": 0.3,
    "epochs": 200,
    "es_patience": 12,
    "label_smoothing": 0.05,
    "grad_clip_norm": 1.0,
    "scheduler_factor": 0.5,
    "scheduler_patience": 4,
    "seed": 2063,
    "num_workers": 0,
}

# Seed fijo: reproduce la corrida ganadora lo mejor posible.
def set_seed(seed=2063):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(BEST_HPARAMS["seed"])

In [56]:
# Función para loguear una figura matplotlib en TensorBoard
def plot_to_tensorboard(fig, writer, tag, step):
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    image = Image.open(buf).convert("RGB")
    image = np.array(image)
    image = torch.tensor(image).permute(2, 0, 1) / 255.0
    writer.add_image(tag, image, global_step=step)
    plt.close(fig)

In [57]:
# Reportes solo al final: evita ruido y warnings por época.
def log_final_classification_report(y_true, y_pred, class_names, writer, prefix="val_final"):
    cm = confusion_matrix(y_true, y_pred)
    fig_cm, ax = plt.subplots(figsize=(8, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45)
    ax.set_title(f"{prefix} - Confusion Matrix")

    fig_path = f"confusion_matrix_{prefix}.png"
    fig_cm.savefig(fig_path, bbox_inches="tight")
    mlflow.log_artifact(fig_path)
    plot_to_tensorboard(fig_cm, writer, f"{prefix}/confusion_matrix", step=0)
    os.remove(fig_path)

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0,
    )
    print(report)
    writer.add_text(f"{prefix}/classification_report", f"<pre>{report}</pre>", 0)

    report_path = f"classification_report_{prefix}.txt"
    with open(report_path, "w") as f:
        f.write(report)
    mlflow.log_artifact(report_path)
    os.remove(report_path)


In [58]:
# Logs separados para esta corrida con HP optimizados.
log_dir = "runs/mlp_trial53_final"
writer = SummaryWriter(log_dir=log_dir)


In [59]:
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None, class_to_idx=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        if class_to_idx is None:
            class_names = sorted([
                d for d in os.listdir(root_dir)
                if os.path.isdir(os.path.join(root_dir, d))
            ])
            self.class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}
        else:
            self.class_to_idx = class_to_idx
            class_names = list(class_to_idx.keys())

        self.classes = class_names

        for cls in class_names:
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_dir):
                continue
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(cls_dir, fname))
                    self.labels.append(self.class_to_idx[cls])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, label


In [60]:
# Augmentations del mejor trial: HFlip, VFlip y brillo/contraste.
train_transform = A.Compose([
    A.Resize(BEST_HPARAMS["input_size"], BEST_HPARAMS["input_size"]),
    A.HorizontalFlip(p=BEST_HPARAMS["HFlip"]),
    A.VerticalFlip(p=BEST_HPARAMS["VFlip"]),
    A.RandomBrightnessContrast(p=BEST_HPARAMS["RBContrast"]),
    A.Normalize(),
    ToTensorV2(),
])


In [61]:
# Validación sin augmentation: mide performance real.
val_test_transform = A.Compose([
    A.Resize(BEST_HPARAMS["input_size"], BEST_HPARAMS["input_size"]),
    A.Normalize(),
    ToTensorV2(),
])


In [62]:
# Paths del split limpio.
train_dir = "data/Split_smol/train"
val_dir = "data/Split_smol/val"


In [63]:
# Val usa el mismo class_to_idx que train: evita label encoding inconsistente.
train_dataset = CustomImageDataset(train_dir, transform=train_transform)
val_dataset = CustomImageDataset(
    val_dir,
    transform=val_test_transform,
    class_to_idx=train_dataset.class_to_idx,
)

assert train_dataset.classes == val_dataset.classes, "Train y val tienen distinto orden de clases."

batch_size = BEST_HPARAMS["batch_size"]
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=BEST_HPARAMS["num_workers"],
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=BEST_HPARAMS["num_workers"],
)

print("Clases:", train_dataset.classes)
print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))


Clases: ['Actinic keratosis', 'Atopic Dermatitis', 'Benign keratosis', 'Dermatofibroma', 'Melanocytic nevus', 'Melanoma', 'Squamous cell carcinoma', 'Tinea Ringworm Candidiasis', 'Vascular lesion']
Train samples: 679
Val samples: 165


In [64]:
class MLPClassifier(nn.Module):
    def __init__(self, input_size, num_classes, hidden_sizes=(1024, 512), dropout=0.35):
        super().__init__()

        layers = [nn.Flatten()]
        in_features = input_size

        for hidden in hidden_sizes:
            layers.extend([
                nn.Linear(in_features, hidden),
                nn.LayerNorm(hidden),      # Estabiliza activaciones del MLP.
                nn.GELU(),                 # Activación suave.
                nn.Dropout(dropout),       # Regularización.
            ])
            in_features = hidden

        layers.append(nn.Linear(in_features, num_classes))
        self.model = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, x):
        return self.model(x)


In [65]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(train_dataset.classes)
input_dim = BEST_HPARAMS["input_size"] ** 2 * 3

model = MLPClassifier(
    input_size=input_dim,
    num_classes=num_classes,
    hidden_sizes=BEST_HPARAMS["hidden_sizes"],
    dropout=BEST_HPARAMS["dropout"],
).to(device)

# Pesos por clase: reduce colapso hacia clases mayoritarias.
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=np.array(train_dataset.labels),
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=BEST_HPARAMS["label_smoothing"],
)

# AdamW + weight decay del mejor trial.
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=BEST_HPARAMS["lr"],
    weight_decay=BEST_HPARAMS["weight_decay"],
)

# Baja LR cuando macro F1 deja de mejorar.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=BEST_HPARAMS["scheduler_factor"],
    patience=BEST_HPARAMS["scheduler_patience"],
)

count_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Device:", device)
print("Trainable params:", count_params)


Device: cuda
Trainable params: 43660809


In [66]:
# Evalúa accuracy, macro F1 y balanced accuracy.
def evaluate(model, loader, criterion, epoch=None, prefix="val"):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)

            loss_sum += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            if i == 0 and epoch is not None:
                img_grid = vutils.make_grid(images[:8].cpu(), normalize=True)
                writer.add_image(f"{prefix}/images", img_grid, global_step=epoch)

    avg_loss = loss_sum / len(loader)
    acc = correct / total
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)

    if epoch is not None:
        writer.add_scalar(f"{prefix}/loss", avg_loss, epoch)
        writer.add_scalar(f"{prefix}/accuracy", acc, epoch)
        writer.add_scalar(f"{prefix}/macro_f1", macro_f1, epoch)
        writer.add_scalar(f"{prefix}/balanced_accuracy", bal_acc, epoch)

    return avg_loss, acc, macro_f1, bal_acc, all_labels, all_preds


In [67]:
# Entrena el MLP final con los HP del trial 53.
n_epochs = BEST_HPARAMS["epochs"]
es_patience = BEST_HPARAMS["es_patience"]

best_state = None
best_epoch = -1
best_val_macro_f1 = -1.0
best_metrics = {}

run_name = "MLP_final_trial53_HP"
with mlflow.start_run(run_name=run_name):
    mlflow.log_params({
        "model": "MLPClassifier",
        "loss_fn": "CrossEntropyLoss_weighted_label_smoothing",
        "train_dir": train_dir,
        "val_dir": val_dir,
        "num_classes": num_classes,
        "train_samples": len(train_dataset),
        "val_samples": len(val_dataset),
        "count_params": count_params,
        **{k: str(v) if isinstance(v, tuple) else v for k, v in BEST_HPARAMS.items()},
    })

    pbar = tqdm(range(n_epochs), desc="MLP final")
    for epoch in pbar:
        model.train()
        running_loss = 0.0
        correct, total = 0, 0
        train_preds, train_labels = [], []

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()

            # Clipping: evita updates explosivos.
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=BEST_HPARAMS["grad_clip_norm"],
            )
            optimizer.step()

            running_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            train_preds.extend(preds.detach().cpu().numpy())
            train_labels.extend(labels.detach().cpu().numpy())

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total
        train_macro_f1 = f1_score(train_labels, train_preds, average="macro", zero_division=0)

        val_loss, val_acc, val_macro_f1, val_bal_acc, y_true, y_pred = evaluate(
            model,
            val_loader,
            criterion,
            epoch=epoch,
            prefix="val",
        )

        scheduler.step(val_macro_f1)

        mlflow.log_metrics({
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "train_macro_f1": train_macro_f1,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "val_macro_f1": val_macro_f1,
            "val_balanced_accuracy": val_bal_acc,
            "lr": optimizer.param_groups[0]["lr"],
        }, step=epoch)

        writer.add_scalar("train/loss", train_loss, epoch)
        writer.add_scalar("train/accuracy", train_acc, epoch)
        writer.add_scalar("train/macro_f1", train_macro_f1, epoch)
        writer.add_scalar("train/lr", optimizer.param_groups[0]["lr"], epoch)

        pbar.set_postfix({
            "val_f1": f"{val_macro_f1:.3f}",
            "val_acc": f"{val_acc:.3f}",
            "best_f1": f"{best_val_macro_f1:.3f}",
        })

        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            best_metrics = {
                "best_epoch": epoch,
                "best_train_loss": train_loss,
                "best_train_accuracy": train_acc,
                "best_train_macro_f1": train_macro_f1,
                "best_val_loss": val_loss,
                "best_val_accuracy": val_acc,
                "best_val_macro_f1": val_macro_f1,
                "best_val_balanced_accuracy": val_bal_acc,
            }
        elif epoch > best_epoch + es_patience:
            tqdm.write(
                f"Early stopping en epoch {epoch + 1}/{n_epochs}. "
                f"Best epoch={best_epoch + 1}, best val macro F1={best_val_macro_f1:.4f}"
            )
            break

    # Recupera el mejor checkpoint antes de guardar.
    model.load_state_dict(best_state)

    final_val_loss, final_val_acc, final_val_macro_f1, final_val_bal_acc, y_true, y_pred = evaluate(
        model,
        val_loader,
        criterion,
        epoch=None,
        prefix="val_final",
    )

    log_final_classification_report(y_true, y_pred, train_dataset.classes, writer, prefix="val_final")

    mlflow.log_metrics(best_metrics)
    mlflow.log_metrics({
        "final_val_loss": final_val_loss,
        "final_val_accuracy": final_val_acc,
        "final_val_macro_f1": final_val_macro_f1,
        "final_val_balanced_accuracy": final_val_bal_acc,
    })

    model_path = "mlp_model_trial53_best.pth"
    torch.save({
        "model_state_dict": model.state_dict(),
        "class_to_idx": train_dataset.class_to_idx,
        "classes": train_dataset.classes,
        "hparams": BEST_HPARAMS,
        "best_metrics": best_metrics,
    }, model_path)
    mlflow.log_artifact(model_path)

    # input_example en numpy: evita warning de MLflow por torch.Tensor.
    example_batch, _ = next(iter(val_loader))
    input_example = example_batch[:1].cpu().numpy()
    try:
        mlflow.pytorch.log_model(
            model,
            artifact_path="pytorch_model",
            input_example=input_example,
            pip_requirements=["torch", "cloudpickle"],
        )
    except Exception as exc:
        print("MLflow no pudo inferir input_example; logueo sin ejemplo.", repr(exc))
        mlflow.pytorch.log_model(
            model,
            artifact_path="pytorch_model",
            pip_requirements=["torch", "cloudpickle"],
        )

print("Mejor epoch:", best_metrics["best_epoch"])
print("Best val macro F1:", best_metrics["best_val_macro_f1"])
print("Best val accuracy:", best_metrics["best_val_accuracy"])
print("Modelo guardado como 'mlp_model_trial53_best.pth'")


MLP final:  26%|██▌       | 52/200 [02:26<06:55,  2.81s/it, val_f1=0.576, val_acc=0.576, best_f1=0.604]


Early stopping en epoch 53/200. Best epoch=40, best val macro F1=0.6041
                            precision    recall  f1-score   support

         Actinic keratosis       0.38      0.55      0.45        20
         Atopic Dermatitis       0.73      0.73      0.73        11
          Benign keratosis       0.86      0.90      0.88        20
            Dermatofibroma       0.50      0.25      0.33        20
         Melanocytic nevus       0.67      0.80      0.73        20
                  Melanoma       0.45      0.45      0.45        20
   Squamous cell carcinoma       0.37      0.35      0.36        20
Tinea Ringworm Candidiasis       0.75      0.64      0.69        14
           Vascular lesion       0.84      0.80      0.82        20

                  accuracy                           0.60       165
                 macro avg       0.62      0.61      0.60       165
              weighted avg       0.60      0.60      0.59       165



2026/06/21 17:22:45 WARNING mlflow.models.signature: Failed to infer the model signature from the input example. Reason: RuntimeError('Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument mat1 in method wrapper_CUDA_addmm)'). To see the full traceback, set the logging level to DEBUG via `logging.getLogger("mlflow").setLevel(logging.DEBUG)`.
2026/06/21 17:22:45 WARNING mlflow.models.model: Failed to validate serving input example {
  "inputs": [
    [
      [
        [
          -0.18280678987503052,
          -0.4739276170730591,
          -0.35405433177948,
          -0.11430778354406357,
          -0.14855729043483734,
          -1.6726603507995605,
          -0.09718302637338638,
          -0.26843056082725525,
          -0.336929589509964,
          -0.09718302637338638,
          0.09118925780057907,
          0.1768130213022232,
          0.1768130213022232,
          -0.4396781027317047,
          0.0

Mejor epoch: 39
Best val macro F1: 0.6040780035553555
Best val accuracy: 0.6
Modelo guardado como 'mlp_model_trial53_best.pth'


In [68]:
# Ejecutar en terminal si estás en VSCode/Jupyter:
# tensorboard --logdir=runs/mlp_trial53_final
